In [1]:
!pip install chembl_webresource_client pandas rdkit scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 42.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00


# LIBRARY IMPORTS

In [2]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import warnings
warnings.filterwarnings("ignore")
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

# DATA INGESTION

In [3]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# ============================================
# FETCH ChEMBL ACTIVITY DATA
# ============================================

activity = new_client.activity

query = activity.filter(
    target_chembl_id="CHEMBL203",
    standard_type="IC50",
    standard_relation="="
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "assay_type",
    "assay_chembl_id",
    "confidence_score"
])

records = []

for i, rec in enumerate(query):

    records.append(rec)

    if i % 1000 == 0:
        print(f"Downloaded {i} records")

# ============================================
# CREATE DATAFRAME
# ============================================

data = pd.DataFrame(records)

print("Dataset Shape:", data.shape)

# ============================================
# SAVE TO EXCEL IN KAGGLE WORKING DIRECTORY
# ============================================

save_path = "/kaggle/working/EGFR_CHEMBL203_raw_data.xlsx"

data.to_excel(save_path, index=False)

print(f"\nExcel file saved at:\n{save_path}")

Downloaded 0 records
Downloaded 1000 records
Downloaded 2000 records
Downloaded 3000 records
Downloaded 4000 records
Downloaded 5000 records
Downloaded 6000 records
Downloaded 7000 records
Downloaded 8000 records
Downloaded 9000 records
Downloaded 10000 records
Downloaded 11000 records
Downloaded 12000 records
Downloaded 13000 records
Downloaded 14000 records
Downloaded 15000 records
Downloaded 16000 records
Downloaded 17000 records
Downloaded 18000 records
Dataset Shape: (18988, 9)

Excel file saved at:
/kaggle/working/EGFR_CHEMBL203_raw_data.xlsx


In [5]:
print(data.columns)

Index(['assay_chembl_id', 'assay_type', 'canonical_smiles',
       'molecule_chembl_id', 'standard_units', 'standard_value', 'type',
       'units', 'value'],
      dtype='object')


# DATA CLEANING

In [6]:
file_path = "/kaggle/working/EGFR_CHEMBL203_raw_data.xlsx"

data = pd.read_excel(file_path)

print("Original Shape:", data.shape)

# ============================================
# KEEP REQUIRED COLUMNS
# ============================================

required_columns = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "assay_type",
    "assay_chembl_id"
]

data = data[required_columns]

# ============================================
# REMOVE MISSING VALUES
# ============================================

data = data.dropna(
    subset=["canonical_smiles", "standard_value","assay_type"]
)

# ============================================
# CONVERT ACTIVITY VALUES TO NUMERIC
# ============================================

data["standard_value"] = pd.to_numeric(
    data["standard_value"],
    errors="coerce"
)

data = data.dropna(subset=["standard_value"])

print("After basic cleaning:", data.shape)

Original Shape: (18988, 9)
After basic cleaning: (18968, 6)


# STANDARDIZATION

In [7]:
allowed_atoms = {
    "H", "C", "N", "O", "S",
    "P", "F", "Cl", "Br", "I"
}

normalizer = rdMolStandardize.Normalizer()
largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()


def standardize_smiles(smiles):

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        # Normalize
        mol = normalizer.normalize(mol)

        # Remove salts / keep largest fragment
        mol = largest_fragment_chooser.choose(mol)

        # Neutralize charges
        mol = uncharger.uncharge(mol)

        # Canonical tautomer normalization
        mol = tautomer_enumerator.Canonicalize(mol)

        # Molecular weight filtering
        mw = Descriptors.MolWt(mol)

        if mw < 100 or mw > 1000:
            return None

        # Allowed atom filtering
        atoms = {
            atom.GetSymbol()
            for atom in mol.GetAtoms()
        }

        if not atoms.issubset(allowed_atoms):
            return None

        # Canonical SMILES
        return Chem.MolToSmiles(
            mol,
            canonical=True
        )

    except:
        return None


# ============================================
# APPLY STANDARDIZATION
# ============================================

data["standardized_smiles"] = data[
    "canonical_smiles"
].apply(standardize_smiles)

# Remove failed molecules
data = data.dropna(
    subset=["standardized_smiles"]
)

print("After SMILES standardization:", data.shape)

[09:36:51] Initializing Normalizer
[09:36:51] Running Normalizer
[09:36:51] Running LargestFragmentChooser
[09:36:51] Running Uncharger
[09:36:51] Running Normalizer
[09:36:51] Running LargestFragmentChooser
[09:36:51] Running Uncharger
[09:36:51] Running Normalizer
[09:36:51] Running LargestFragmentChooser
[09:36:51] Running Uncharger
[09:36:52] Running Normalizer
[09:36:52] Running LargestFragmentChooser
[09:36:52] Running Uncharger
[09:36:52] Tautomer enumeration stopped at 156 tautomers: max transforms reached
[09:36:52] Running Normalizer
[09:36:52] Running LargestFragmentChooser
[09:36:52] Running Uncharger
[09:36:53] Tautomer enumeration stopped at 156 tautomers: max transforms reached
[09:36:53] Running Normalizer
[09:36:53] Running LargestFragmentChooser
[09:36:53] Running Uncharger
[09:36:53] Tautomer enumeration stopped at 156 tautomers: max transforms reached
[09:36:54] Running Normalizer
[09:36:54] Running LargestFragmentChooser
[09:36:54] Running Uncharger
[09:36:54] Runn

After SMILES standardization: (18764, 7)


[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running Uncharger
[09:53:06] Running Normalizer
[09:53:06] Running LargestFragmentChooser
[09:53:06] Running 

# KEEP ONLY POSITIVE ACTIVITY VALUES


In [8]:
data = data[
    data["standard_value"] > 0
]

print("After removing non-positive IC50:", data.shape)

After removing non-positive IC50: (18764, 7)


# UNIT HARMONIZATION


In [9]:

unit_conversion = {
    "nM": 1,
    "uM": 1000,
    "µM": 1000,
    "mM": 1_000_000
}


def convert_to_nM(row):

    unit = row["standard_units"]
    value = row["standard_value"]

    if unit in unit_conversion:
        return value * unit_conversion[unit]

    return np.nan


data["IC50_nM"] = data.apply(
    convert_to_nM,
    axis=1
)

data = data.dropna(
    subset=["IC50_nM"]
)

print("After unit harmonization:", data.shape)

# ============================================
# pIC50 CALCULATION
# ============================================

data["IC50_M"] = data["IC50_nM"] * 1e-9

data["pIC50"] = -np.log10(
    data["IC50_M"]
)

# ============================================
# BINARY ACTIVITY LABEL
# Active if pIC50 >= 6
# ============================================

data["activity_class"] = data[
    "pIC50"
].apply(
    lambda x: 1 if x >= 6 else 0
)

After unit harmonization: (18693, 8)


# DUPLICATE HANDLING

In [10]:
from scipy.stats.mstats import gmean

def geometric_mean_pic50(values):

    values = np.array(values)

    # Convert pIC50 → IC50 molar
    ic50_values = 10 ** (-values)

    # Geometric mean IC50
    geo_ic50 = gmean(ic50_values)

    # Convert back → pIC50
    return -np.log10(geo_ic50)


data = (
    data.groupby(
        [
            "molecule_chembl_id",
            "standardized_smiles"
        ],
        as_index=False
    )
    .agg({
        "IC50_nM": lambda x: gmean(x),
        "pIC50": geometric_mean_pic50,
        "activity_class": "max",
        "assay_type": "first",
        "assay_chembl_id": "first"
    })
)

print("After duplicate handling:", data.shape)

After duplicate handling: (10483, 7)


# GENERATE MORGAN FINGERPRINTS
## For activity cliff detection

In [11]:
# =====================================================
# GENERATE MORGAN FINGERPRINTS
# =====================================================

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

import random

print("\nGenerating fingerprints...")


def generate_fingerprint(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )


data["fingerprint"] = data[
    "standardized_smiles"
].apply(generate_fingerprint)

data = data.dropna(
    subset=["fingerprint"]
).reset_index(drop=True)

print(
    "After fingerprint generation:",
    data.shape
)


# =====================================================
# OPTIMIZED ACTIVITY CLIFF DETECTION
# Methodology:
# similarity >= 0.85
# delta_pIC50 >= 2
#
# Optimization:
# compare against random subset
# instead of full O(n²) search
# =====================================================

print("\nDetecting activity cliffs...")

activity_cliff_indices = set()

fingerprints = data["fingerprint"].tolist()
pic50_values = data["pIC50"].tolist()

num_molecules = len(data)

# ---------------------------------------------
# IMPORTANT OPTIMIZATION
# ---------------------------------------------
# Instead of comparing every pair,
# compare each molecule against
# a random subset
# ---------------------------------------------

MAX_COMPARISONS_PER_MOLECULE = 200

random.seed(42)

for i in range(num_molecules):

    # Create candidate indices excluding self
    candidate_indices = list(
        range(i + 1, num_molecules)
    )

    # Skip if no candidates left
    if len(candidate_indices) == 0:
        continue

    # Random subset sampling
    sampled_indices = random.sample(
        candidate_indices,
        min(
            MAX_COMPARISONS_PER_MOLECULE,
            len(candidate_indices)
        )
    )

    for j in sampled_indices:

        similarity = TanimotoSimilarity(
            fingerprints[i],
            fingerprints[j]
        )

        # Fast reject
        if similarity < 0.85:
            continue

        delta_pic50 = abs(
            pic50_values[i]
            - pic50_values[j]
        )

        if delta_pic50 >= 2:

            activity_cliff_indices.add(i)
            activity_cliff_indices.add(j)

    # Progress tracking
    if i % 1000 == 0:

        print(
            f"Processed "
            f"{i}/{num_molecules}"
        )


print(
    "\nActivity cliff molecules detected:",
    len(activity_cliff_indices)
)


# =====================================================
# ACTIVITY CLIFF FLAGGING
# =====================================================

print("\nFlagging activity cliffs...")

data["activity_cliff"] = False

if len(activity_cliff_indices) > 0:

    data.loc[
        list(activity_cliff_indices),
        "activity_cliff"
    ] = True


print("\nActivity Cliff Distribution:")

print(
    data["activity_cliff"]
    .value_counts()
)


# =====================================================
# REMOVAL
# =====================================================

REMOVE_ACTIVITY_CLIFFS = False

if REMOVE_ACTIVITY_CLIFFS:

    data = data[
        data["activity_cliff"] == False
    ].reset_index(drop=True)

    print(
        "\nAfter activity cliff removal:",
        data.shape
    )


Generating fingerprints...


[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerator
[09:59:33] DEPRECATION WARNING: please use MorganGenerat

After fingerprint generation: (10483, 8)

Detecting activity cliffs...
Processed 0/10483
Processed 1000/10483
Processed 2000/10483
Processed 3000/10483
Processed 4000/10483
Processed 5000/10483
Processed 6000/10483
Processed 7000/10483
Processed 8000/10483
Processed 9000/10483
Processed 10000/10483

Activity cliff molecules detected: 20

Flagging activity cliffs...

Activity Cliff Distribution:
activity_cliff
False    10463
True        20
Name: count, dtype: int64


# CLASS BALANCE ANALYSIS

In [12]:
class_counts = data[
    "activity_class"
].value_counts()

print("\nClass Distribution:")
print(class_counts)

majority = class_counts.max()
minority = class_counts.min()

imbalance_ratio = majority / minority

print(
    f"\nClass imbalance ratio: "
    f"{imbalance_ratio:.2f}:1"
)


# =====================================================
# CLASS WEIGHTING DECISION
# Methodology:
# use class weighting if imbalance > 4:1
# =====================================================

print("\nCLASS WEIGHTING DECISION")

USE_CLASS_WEIGHTS = False

if imbalance_ratio > 4:

    USE_CLASS_WEIGHTS = True

    print(
        "WARNING: imbalance exceeds 4:1"
    )

    print(
        "Class weighting recommended."
    )

else:

    print(
        "Class balance acceptable."
    )


Class Distribution:
activity_class
1    7944
0    2539
Name: count, dtype: int64

Class imbalance ratio: 3.13:1

CLASS WEIGHTING DECISION
Class balance acceptable.


#  GENERATE MURCKO SCAFFOLDS

In [13]:
def generate_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol
    )


data["scaffold"] = data[
    "standardized_smiles"
].apply(generate_scaffold)

data = data.dropna(
    subset=["scaffold"]
).reset_index(drop=True)

print(
    "After scaffold generation:",
    data.shape
)


# =====================================================
# SCAFFOLD FREQUENCY ANALYSIS
# =====================================================

print("\nSCAFFOLD FREQUENCY ANALYSIS")

scaffold_counts = (
    data["scaffold"]
    .value_counts()
)

sorted_scaffolds = (
    scaffold_counts.index.tolist()
)

print(
    "Unique scaffolds:",
    len(sorted_scaffolds)
)


# =====================================================
# FREQUENCY-AWARE
# SCAFFOLD SPLITTING
# =====================================================

print(
    "\n SCAFFOLD-BASED SPLITTING"
)

TRAIN_FRAC = 0.70
VALID_FRAC = 0.10
TEST_FRAC = 0.20

train_scaffolds = []
valid_scaffolds = []
test_scaffolds = []

train_count = 0
valid_count = 0
test_count = 0

total_molecules = len(data)

train_target = TRAIN_FRAC * total_molecules
valid_target = VALID_FRAC * total_molecules
test_target = TEST_FRAC * total_molecules


for scaffold in sorted_scaffolds:

    scaffold_size = scaffold_counts[
        scaffold
    ]

    if train_count < train_target:

        train_scaffolds.append(
            scaffold
        )

        train_count += scaffold_size

    elif valid_count < valid_target:

        valid_scaffolds.append(
            scaffold
        )

        valid_count += scaffold_size

    else:

        test_scaffolds.append(
            scaffold
        )

        test_count += scaffold_size

After scaffold generation: (10483, 10)

SCAFFOLD FREQUENCY ANALYSIS
Unique scaffolds: 3686

 SCAFFOLD-BASED SPLITTING


# CREATE SPLITS


In [14]:
train_df = data[
    data["scaffold"].isin(
        train_scaffolds
    )
].reset_index(drop=True)

valid_df = data[
    data["scaffold"].isin(
        valid_scaffolds
    )
].reset_index(drop=True)

test_df = data[
    data["scaffold"].isin(
        test_scaffolds
    )
].reset_index(drop=True)


print("\nDATA SPLITS")

print(
    "Train:",
    train_df.shape
)

print(
    "Validation:",
    valid_df.shape
)

print(
    "Test:",
    test_df.shape
)


# =====================================================
# SCAFFOLD LEAKAGE CHECK
# =====================================================

print(
    "\nCHECKING "
    "SCAFFOLD LEAKAGE"
)

train_set = set(
    train_df["scaffold"]
)

valid_set = set(
    valid_df["scaffold"]
)

test_set = set(
    test_df["scaffold"]
)

assert len(
    train_set.intersection(valid_set)
) == 0

assert len(
    train_set.intersection(test_set)
) == 0

assert len(
    valid_set.intersection(test_set)
) == 0

print(
    "No scaffold leakage detected."
)


# =====================================================
# SCAFFOLD NOVELTY METRIC
# =====================================================

print(
    "\nSCAFFOLD "
    "NOVELTY METRIC"
)

unseen_test_scaffolds = (
    test_set - train_set
)

scaffold_novelty = (
    len(unseen_test_scaffolds)
    / len(test_set)
) * 100

print(
    f"Scaffold novelty "
    f"in test set: "
    f"{scaffold_novelty:.2f}%"
)


# =====================================================
#  REMOVE TEMP COLUMNS
# =====================================================

train_df = train_df.drop(
    columns=["fingerprint"]
)

valid_df = valid_df.drop(
    columns=["fingerprint"]
)

test_df = test_df.drop(
    columns=["fingerprint"]
)


DATA SPLITS
Train: (7339, 10)
Validation: (1049, 10)
Test: (2095, 10)

CHECKING SCAFFOLD LEAKAGE
No scaffold leakage detected.

SCAFFOLD NOVELTY METRIC
Scaffold novelty in test set: 100.00%


In [16]:
import os
print(
    "\nSAVING "
    "CLEANED DATASET"
)

cleaned_file = (
    "/kaggle/working/"
    "EGFR_CHEMBL203_cleaned_data.xlsx"
)

data.to_excel(
    cleaned_file,
    index=False
)

print(
    "Cleaned dataset saved:"
)

print(cleaned_file)


# =====================================================
# STEP 14 — SAVE SPLITS
# =====================================================

print(
    "\nSAVING SPLITS"
)

split_file = (
    "/kaggle/working/"
    "EGFR_CHEMBL203_split_data.xlsx"
)

with pd.ExcelWriter(
    split_file
) as writer:

    train_df.to_excel(
        writer,
        sheet_name="Train",
        index=False
    )

    valid_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )

print(
    "Split datasets saved:"
)

print(split_file)


# =====================================================
# STEP 15 — CLASS DISTRIBUTION
# ACROSS SPLITS
# =====================================================

print(
    "\nCLASS "
    "DISTRIBUTIONS"
)

print("\nTRAIN")
print(
    train_df[
        "activity_class"
    ].value_counts()
)

print("\nVALIDATION")
print(
    valid_df[
        "activity_class"
    ].value_counts()
)

print("\nTEST")
print(
    test_df[
        "activity_class"
    ].value_counts()
)


# =====================================================
# FILE CHECK
# =====================================================

print(
    "\nFILE CHECK"
)

print(
    os.listdir(
        "/kaggle/working/"
    )
)


# =====================================================
# FINAL SUMMARY
# =====================================================

print("\nFINAL SUMMARY")
print("=" * 50)

print(
    "Final dataset size:",
    len(data)
)

print(
    "Train molecules:",
    len(train_df)
)

print(
    "Validation molecules:",
    len(valid_df)
)

print(
    "Test molecules:",
    len(test_df)
)

print(
    "Activity cliff molecules:",
    len(activity_cliff_indices)
)

print(
    f"Imbalance ratio: "
    f"{imbalance_ratio:.2f}:1"
)

print(
    f"Scaffold novelty: "
    f"{scaffold_novelty:.2f}%"
)

print(
    "Use class weights:",
    USE_CLASS_WEIGHTS
)


SAVING CLEANED DATASET
Cleaned dataset saved:
/kaggle/working/EGFR_CHEMBL203_cleaned_data.xlsx

SAVING SPLITS
Split datasets saved:
/kaggle/working/EGFR_CHEMBL203_split_data.xlsx

CLASS DISTRIBUTIONS

TRAIN
activity_class
1    5603
0    1736
Name: count, dtype: int64

VALIDATION
activity_class
1    817
0    232
Name: count, dtype: int64

TEST
activity_class
1    1524
0     571
Name: count, dtype: int64

FILE CHECK
['EGFR_CHEMBL203_split_data.xlsx', 'EGFR_CHEMBL203_raw_data.xlsx', 'EGFR_CHEMBL203_cleaned_data.xlsx', '.virtual_documents']

FINAL SUMMARY
Final dataset size: 10483
Train molecules: 7339
Validation molecules: 1049
Test molecules: 2095
Activity cliff molecules: 20
Imbalance ratio: 3.13:1
Scaffold novelty: 100.00%
Use class weights: False


In [ ]:
'''import pandas as pd

df = pd.read_excel("/kaggle/input/datasets/sherongeorge/egfr-dataset/EGFR_CHEMBL203_cleaned_data.xlsx")
df.head()'''

In [ ]:
'''from sklearn.model_selection import train_test_split

train_df, external_df = train_test_split(df, test_size=0.3, random_state=42)

print("Train:", train_df.shape)
print("External (your dataset):", external_df.shape)'''

In [ ]:
#external_df.to_csv("bindingdb_clean.csv", index=False)

In [ ]:
'''external_df = external_df.rename(columns={
    'clean_smiles': 'SMILES'
})'''

In [ ]:
'''from rdkit import Chem
import pickle

def mol_to_graph_simple(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    nodes = []
    for atom in mol.GetAtoms():
        nodes.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic())
        ])

    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append((i, j))
        edges.append((j, i))

    return {
        "nodes": nodes,
        "edges": edges,
        "label": label
    }
external_df = external_df.rename(columns={
    'clean_smiles': 'SMILES'
})

graph_data = []

for _, row in external_df.iterrows():
    g = mol_to_graph_simple(row['SMILES'], row['pIC50'])
    graph_data.append(g)

# Save
with open("bindingdb_graph.pkl", "wb") as f:
    pickle.dump(graph_data, f)

print("Graph dataset saved successfully")'''